# 🧭 Notebook 3: When to use CQRS (and when not to)

CQRS is powerful — but it isn't free. This notebook covers the **trade-offs**,
the **eventual consistency** surprise, and a small **real-world flavoured**
example: an online store with a **checkout service** (writes) and an
**analytics dashboard** (reads).

## 🛠️ Setup

```bash
cd 05-microservices/cqrs
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## ✅ Use CQRS when…

- Read load is **much higher** than write load (e-commerce storefront,
  social feed, analytics dashboards).
- Reads need a shape very different from writes (search, reports, timelines).
- You need **audit / history** (often pair CQRS with event sourcing).
- Teams want to scale read/write sides **independently**.

## 🚫 Avoid CQRS when…

- The domain is simple CRUD — a single model is easier and correct.
- Your team is small and the extra moving parts would slow you down.
- Strong, immediate consistency is a hard requirement across **every** read
  (banking "balance right now", stock trading). You *can* still use CQRS, but
  you'll need read-your-writes tricks and the complexity usually isn't worth it.

## 🧩 Typical architecture

```
 Client ──► API ──► Command Handler ──► Write DB ──► Event Bus
                                                       │
                                                       ▼
                                            Projector(s) ──► Read DB(s)
 Client ──► API ──► Query Handler  ◄───── Read DB
```

Each arrow can cross a network. Every network hop is a place where things can
be delayed, retried, or delivered out-of-order. Welcome to **eventual consistency**.


## 1️⃣ Eventual consistency in action

We'll simulate a small lag between the command side and the projection (as if
an event bus took a moment to deliver). Readers during that window see **stale**
data — that is the defining property of CQRS you must plan for.

In [1]:
import queue, threading, time

event_bus = queue.Queue()

# Write side
balances_write = {'alice': 100.0}

def deposit(user, amount):
    balances_write[user] = balances_write.get(user, 0) + amount
    event_bus.put({'type': 'Deposited', 'user': user, 'amount': amount})

# Read side (projection) - runs in a background thread with a small lag
balances_read = {'alice': 100.0}
stop = threading.Event()

def projector():
    while not stop.is_set():
        try:
            e = event_bus.get(timeout=0.1)
        except queue.Empty:
            continue
        time.sleep(0.2)  # pretend the bus / projection is slow
        if e['type'] == 'Deposited':
            balances_read[e['user']] = balances_read.get(e['user'], 0) + e['amount']

t = threading.Thread(target=projector, daemon=True)
t.start()

# User deposits $50, then immediately reads their balance.
deposit('alice', 50)
print('immediately after deposit (read side):', balances_read['alice'])  # stale!

time.sleep(0.5)
print('a moment later                       :', balances_read['alice'])  # caught up

stop.set(); t.join(timeout=1)


immediately after deposit (read side): 100.0


a moment later                       : 150.0


### Strategies to cope with the stale window

1. **UI hints:** "Your order was placed; dashboard will update in a few seconds."
2. **Read-your-writes:** after a write, read from the write side (or pass the
   expected version in the query and have the read side wait for it).
3. **Optimistic UI:** update the screen locally from the command's response,
   the projection will catch up.
4. **Stronger bus guarantees:** ordered delivery per key (Kafka partitions,
   Postgres logical replication) keeps projections causally consistent.


## 1️⃣.5 Read-your-writes: *did my write land yet?*

A common UX problem with CQRS: a user clicks "Save", the write succeeds, they
refresh, and the screen still shows the *old* data (projection has not caught
up). One fix is to attach a **version** (monotonic counter) to every write and
have the client ask the read side: *answer only once you have seen version >= N*.


In [2]:
import queue, threading, time

bus = queue.Queue()
write_version = 0                       # monotonic counter on the write side
read_version  = 0                       # latest version the read side applied
balances_w = {'alice': 100.0}
balances_r = {'alice': 100.0}
lock = threading.Condition()

def deposit(user, amount):
    global write_version
    write_version += 1
    balances_w[user] = balances_w.get(user, 0) + amount
    bus.put({'v': write_version, 'user': user, 'amount': amount})
    return write_version  # client remembers "I expect version >= N"

stop = threading.Event()

def projector():
    global read_version
    while not stop.is_set():
        try:
            e = bus.get(timeout=0.1)
        except queue.Empty:
            continue
        time.sleep(0.2)  # pretend the bus is slow
        balances_r[e['user']] = balances_r.get(e['user'], 0) + e['amount']
        with lock:
            read_version = e['v']
            lock.notify_all()

threading.Thread(target=projector, daemon=True).start()

def query_balance(user, min_version, timeout=1.0):
    """Wait until the read side has caught up to min_version, then answer."""
    deadline = time.time() + timeout
    with lock:
        while read_version < min_version and time.time() < deadline:
            lock.wait(timeout=deadline - time.time())
    return balances_r.get(user, 0)

v = deposit('alice', 50)
print(f'write landed at version {v}; querying with min_version={v}...')
print('balance (read-your-writes):', query_balance('alice', v))

stop.set()


write landed at version 1; querying with min_version=1...


balance (read-your-writes): 150.0


## 2️⃣ Real-world flavoured example: store + analytics

- **Checkout service** (write): validates and records orders.
- **Analytics service** (read): shows *"today's revenue by product category"*.
- They share an **event log**. The analytics side keeps a tiny projection
  optimised for the dashboard query.

In [3]:
from collections import defaultdict
from datetime import date

event_log = []

# ---- Write side ----
PRODUCT_CATEGORY = {'book': 'media', 'movie': 'media',
                    'shirt': 'apparel', 'hat': 'apparel'}

def checkout(order_id, items):
    '''items: [(product, qty, unit_price)]'''
    for product, qty, price in items:
        if product not in PRODUCT_CATEGORY:
            raise ValueError(f'unknown product {product}')
        if qty <= 0 or price <= 0:
            raise ValueError('qty/price must be positive')
    event_log.append({'type': 'OrderPlaced', 'oid': order_id,
                      'items': items, 'day': str(date.today())})

# ---- Read side: projection for the dashboard ----
revenue_by_category_today = defaultdict(float)

def project(event):
    if event['type'] == 'OrderPlaced' and event['day'] == str(date.today()):
        for product, qty, price in event['items']:
            cat = PRODUCT_CATEGORY[product]
            revenue_by_category_today[cat] += qty * price

# Simulate: orders arrive, projection is updated
checkout(1, [('book', 2, 15.0), ('hat', 1, 20.0)])
checkout(2, [('shirt', 3, 25.0)])
checkout(3, [('movie', 1, 12.5), ('book', 1, 15.0)])

for e in event_log:
    project(e)

def dashboard():
    return dict(revenue_by_category_today)

print("Today's revenue by category:", dashboard())


Today's revenue by category: {'media': 57.5, 'apparel': 95.0}


### What to notice

- The **checkout** code never touches the dashboard structure. It just records facts.
- The **dashboard** code never validates orders. It just reacts to events.
- If tomorrow we need *"top 5 products by revenue"*, we add another projection
  and replay `event_log`. The checkout code doesn't change. 🎉


## 3️⃣ Common pitfalls

| Pitfall | Fix |
|---|---|
| Projections drift from events (someone edited the read DB) | Treat projections as *disposable*: rebuild from events. |
| Event schema changes break old consumers | Version events (`V1`, `V2`), write upcasters, never delete fields. |
| Out-of-order events corrupt projections | Ensure per-entity ordering (partition key) on the bus. |
| Users confused by stale reads | Surface "processing…" in the UI, use read-your-writes for critical paths. |
| CQRS used for a simple CRUD app | Stop. Use one model. Come back when you actually need it. |

## 📚 Further reading

- Martin Fowler — [CQRS](https://martinfowler.com/bliki/CQRS.html)
- Greg Young — *CQRS Documents* (original write-up)
- Microsoft — [CQRS pattern](https://learn.microsoft.com/azure/architecture/patterns/cqrs)
- Event Store — [Event Sourcing basics](https://www.eventstore.com/event-sourcing)
